# 2 · Corpus & speaker statistics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/02_corpus/02_corpus_statistics.ipynb)

**Pipeline stage 2 of 6.** Quantify the digitised corpus: token/type counts,
per-CD distribution, speaker demographics (age, neighbourhood), and the phoneme
inventory used downstream.

Reference figures for *Alles Kölsch*: **~46k word tokens · 125 speakers · 49
Cologne neighbourhoods · 44 phonemes** (≈ 4 h of narrative speech).

## Setup & data

In [ ]:
!pip -q install pandas matplotlib
import pandas as pd, numpy as np, glob, os, re, collections
import matplotlib.pyplot as plt

# Expect a metadata table with one row per recording/speaker.
# Columns: speaker_id, cd, track, age, neighbourhood, transcript (or txt path)
META_CSV = "metadata.csv"          # <- your speaker/recording metadata
OCR_DIR  = "../01_ocr/ocr_txt"     # corrected transcriptions from notebook 1

# Demo fallback so the notebook runs without your private data:
if os.path.exists(META_CSV):
    df = pd.read_csv(META_CSV)
else:
    df = pd.DataFrame({
        "speaker_id":[f"s{i:03d}" for i in range(1,13)],
        "cd":[1,1,2,2,3,3,4,4,1,2,3,4],
        "age":[34,67,52,71,29,63,88,45,58,61,40,77],
        "neighbourhood":["Altstadt-Süd","Ehrenfeld","Nippes","Kalk","Ehrenfeld",
                         "Altstadt-Süd","Nippes","Kalk","Altstadt-Süd","Ehrenfeld",
                         "Sülz","Deutz"],
        "transcript":["un dann hammer dat jemaht"]*12,
    })
    print("(using demo metadata - drop in metadata.csv for real numbers)")
df.head()

## 1 · Token & type counts

In [ ]:
def tokens(text):
    return re.findall(r"\S+", str(text).lower())

all_tokens = [t for tx in df["transcript"] for t in tokens(tx)]
ttypes = set(all_tokens)
print(f"word tokens : {len(all_tokens):,}")
print(f"word types  : {len(ttypes):,}")
print(f"type/token  : {len(ttypes)/max(1,len(all_tokens)):.3f}")

# per-CD distribution
per_cd = (df.assign(n=df['transcript'].map(lambda t: len(tokens(t))))
            .groupby('cd')['n'].sum())
print("\nwords per CD:\n", per_cd.to_string())

## 2 · Speaker demographics

In [ ]:
print("age range :", int(df.age.min()), "-", int(df.age.max()))
print("age mean  :", round(df.age.mean(),1), " median:", df.age.median())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bins = [10,20,30,40,50,60,70,80,90]
ax[0].hist(df.age, bins=bins, color="#0F7C73", edgecolor="white")
ax[0].set_title("Speaker age distribution"); ax[0].set_xlabel("age"); ax[0].set_ylabel("speakers")
nb = df.neighbourhood.value_counts().head(12)
ax[1].barh(nb.index[::-1], nb.values[::-1], color="#1F3864")
ax[1].set_title("Top neighbourhoods"); ax[1].set_xlabel("speakers")
fig.tight_layout(); plt.show()

## 3 · Phoneme inventory & distribution

After phonological normalisation (Notebook 4) the working inventory has **44
phonemes**. Here we tokenise the IPA references and plot the stacked
train/valid/test ratio per phoneme — the same figure used in the paper. (If you
only have orthography at this stage, run Notebook 4 first to get IPA.)

In [ ]:
# Minimal IPA phoneme tokeniser (full version in Notebook 4)
PHONEMES = ['aː','ɛː','eː','iː','oː','uː','yː','øː','aɪ','aʊ','ɔɪ','ɛɪ','ɔʏ','ɐʊ',
 't͡s','p͡f','t͡ʃ','a','ɛ','ɪ','ɔ','ʊ','ʏ','œ','ə','ɐ','e','o','i','u','y','ø',
 'ʃ','ʒ','ç','χ','x','f','v','s','z','h','p','b','t','d','k','ɡ','ʔ','m','n','ŋ',
 'l','ʁ','j','r','w','ɥ']
_P = sorted(PHONEMES, key=len, reverse=True)
def to_phonemes(ipa_word):
    out, i = [], 0
    while i < len(ipa_word):
        for p in _P:
            if ipa_word.startswith(p, i): out.append(p); i += len(p); break
        else: i += 1
    return out

def count_phonemes(series):
    c = collections.Counter()
    for s in series.astype(str):
        for w in s.split():
            c.update(to_phonemes(w))
    return c

# If you have an 'ipa' column with split-level labels per row, group by split:
if "ipa" in df.columns and "split" in df.columns:
    splits = {k: count_phonemes(df[df.split==k]["ipa"]) for k in ["train","valid","test"]}
    phones = sorted(set().union(*[set(c) for c in splits.values()]))
    tot = {k: max(1,sum(c.values())) for k,c in splits.items()}
    x = np.arange(len(phones)); bottom = np.zeros(len(phones))
    fig, ax = plt.subplots(figsize=(16,6))
    for name,color in [("train","#1f77b4"),("valid","#ff7f0e"),("test","#2ca02c")]:
        vals = np.array([splits[name].get(p,0)/tot[name] for p in phones])
        ax.bar(x, vals, bottom=bottom, label=name, color=color); bottom += vals
    ax.set_xticks(x); ax.set_xticklabels(phones, rotation=45)
    ax.set_xlabel("Kölsch phoneme (IPA)"); ax.set_ylabel("Ratio (stacked)")
    ax.set_title(f"Kölsch Phoneme Distribution — {len(phones)} phonemes"); ax.legend()
    fig.tight_layout(); plt.savefig("kolsch_phoneme_distribution.png", dpi=150); plt.show()
else:
    print("Add 'ipa' + 'split' columns (from Notebook 4) to plot the distribution.")

## Summary

This notebook produces the corpus headline numbers (tokens, types, TTR,
per-CD), the speaker demographics figures, and the phoneme distribution chart.
These feed the data sections of the paper and the report.